<a href="https://colab.research.google.com/github/DrakeJay/LameChat/blob/main/LameChatV3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 !pip install tiktoken requests tqdm

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import tiktoken
import math
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
enc = tiktoken.get_encoding("gpt2")   # 50,257-token BPE vocab

vocab_size = enc.n_vocab              # 50,257
print(f"Vocab size (BPE): {vocab_size}")

# Special tokens — GPT-2 encoding already reserves <|endoftext|>
EOT_ID   = enc.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

# We reuse unused token IDs for our chat special tokens
# (GPT-2 vocab has slots 50257+ but tiktoken clips; use rare strings instead)
USER_STR = "USER:"
BOT_STR  = "BOT:"
END_STR  = "END\n"

def encode(s: str) -> list[int]:
    return enc.encode(s, allowed_special={"<|endoftext|>"})

def decode(ids: list[int]) -> str:
    return enc.decode(ids)

# Quick sanity check
test = "Hark, who goes there?"
assert decode(encode(test)) == test
print(f"Tokenizer check passed. '{test}' → {len(encode(test))} tokens")



Vocab size (BPE): 50257
Tokenizer check passed. 'Hark, who goes there?' → 7 tokens


In [4]:
print("Downloading corpora...")

# Tiny Shakespeare
shakes_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
shakes_text = requests.get(shakes_url).text
print(f"  Shakespeare: {len(shakes_text):,} chars")

# A second Gutenberg text — The Complete Works of Jane Austen excerpt
# (public domain, diverse dialogue, conversational tone)
austen_url = "https://www.gutenberg.org/files/1342/1342-0.txt"   # Pride and Prejudice
try:
    austen_text = requests.get(austen_url, timeout=10).text
    # Strip Gutenberg header/footer
    start = austen_text.find("It is a truth")
    end   = austen_text.rfind("End of the Project Gutenberg")
    austen_text = austen_text[start:end] if start != -1 else austen_text[:200_000]
    print(f"  Austen:      {len(austen_text):,} chars")
except Exception:
    austen_text = ""
    print("  Austen:      skipped (network issue)")

combined_text = shakes_text + "\n" + austen_text
print(f"  Combined:    {len(combined_text):,} chars")

# Encode the full corpus
print("Encoding corpus with BPE (may take ~30s)...")
data = torch.tensor(encode(combined_text), dtype=torch.long)
print(f"  Token count: {len(data):,}")

# Train / val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

  Shakespeare: 1,115,394 chars
  Austen:      694,502 chars
  Combined:    1,809,897 chars
Encoding corpus with BPE (may take ~30s)...
  Token count: 516,783


In [5]:
block_size = 256
batch_size = 48        # reduce to 32 if you hit OOM
n_embd     = 384
n_head     = 6
n_layer    = 8         # 8 layers instead of 24: more data-efficient, trains 3x faster
dropout    = 0.1

torch.manual_seed(42)

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)



In [6]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),                      # GELU > ReLU for transformers
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        # weight tying — significantly reduces parameters & improves quality
        self.token_embedding_table.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens,
                 temperature=0.8, top_k=50, top_p=0.92,
                 repetition_penalty=1.15):
        """
        Top-p (nucleus) sampling with top-k filtering and repetition penalty.

        temperature       : higher = more random. 0.7–0.9 is the sweet spot.
        top_k             : keep only the top-k most likely tokens each step.
        top_p             : keep the smallest set of tokens whose cumulative
                            probability exceeds p (nucleus sampling).
        repetition_penalty: > 1.0 divides logits of already-seen tokens,
                            reducing loops and repeated phrases.
        """
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]          # (B, vocab_size)

            # ── Repetition penalty ──────────────────────────────
            if repetition_penalty != 1.0:
                for token_id in set(idx[0].tolist()):
                    if logits[0, token_id] < 0:
                        logits[0, token_id] *= repetition_penalty
                    else:
                        logits[0, token_id] /= repetition_penalty

            # ── Temperature ─────────────────────────────────────
            logits = logits / temperature

            # ── Top-k filter ────────────────────────────────────
            if top_k > 0:
                top_k_vals = torch.topk(logits, min(top_k, logits.size(-1)))[0]
                logits[logits < top_k_vals[:, [-1]]] = float("-inf")

            # ── Top-p (nucleus) filter ───────────────────────────
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                # Remove tokens once cumulative prob exceeds top_p
                sorted_logits[cumulative_probs - F.softmax(sorted_logits, dim=-1) > top_p] = float("-inf")
                # Scatter back to original ordering
                logits = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_logits)

            probs    = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat((idx, idx_next), dim=1)

            # Stop early on end token
            if idx_next.item() == EOT_ID:
                break

        return idx

model = GPTLanguageModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.1f}M")



Model parameters: 33.6M


In [7]:
max_steps   = 5000
warmup_steps = 200
max_lr      = 3e-4
min_lr      = 3e-5
eval_interval = 500

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.01)

def get_lr(step):
    """Cosine decay with linear warmup."""
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    if step > max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ["train", "val"]:
        L = []
        for _ in range(20):
            x, y = get_batch(split)
            _, loss = model(x, y)
            L.append(loss.item())
        losses[split] = sum(L) / len(L)
    model.train()
    return losses

print("Pre-training...")
train_losses, val_losses = [], []

for step in tqdm(range(max_steps), desc="Pre-training"):
    # Update LR
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
    optimizer.step()

    if step % eval_interval == 0 or step == max_steps - 1:
        losses = estimate_loss()
        train_losses.append(losses["train"])
        val_losses.append(losses["val"])
        print(f"  step {step:4d} | train {losses['train']:.4f} | val {losses['val']:.4f} | lr {lr:.2e}")

print("Pre-training complete.")

# Quick generation test
model.eval()
ctx = torch.zeros((1,1), dtype=torch.long, device=device)
sample = decode(model.generate(ctx, max_new_tokens=150)[0].tolist())
print("\nSample output:\n", sample[:400])



Pre-training...


Pre-training:   0%|          | 0/5000 [00:00<?, ?it/s]

  step    0 | train 10.9666 | val 10.9566 | lr 0.00e+00
  step  500 | train 6.0144 | val 6.2757 | lr 2.97e-04
  step 1000 | train 5.2299 | val 5.4698 | lr 2.82e-04
  step 1500 | train 4.8416 | val 5.0608 | lr 2.54e-04
  step 2000 | train 4.5963 | val 4.9235 | lr 2.17e-04
  step 2500 | train 4.2881 | val 4.7044 | lr 1.74e-04
  step 3000 | train 4.0136 | val 4.5104 | lr 1.30e-04
  step 3500 | train 3.8105 | val 4.4452 | lr 9.00e-05
  step 4000 | train 3.7025 | val 4.3842 | lr 5.79e-05
  step 4500 | train 3.5942 | val 4.3757 | lr 3.72e-05
  step 4999 | train 3.5769 | val 4.3634 | lr 3.00e-05
Pre-training complete.

Sample output:
 !
If the time have no more, I'll not stay;
For then they do it so dear and make us.
Thou hastst thou didst give thee, that kill'd me,
And leave thy high hours of this land,
Which to make the crown upon the sea's hands.
Now comes all, or you both so rough?
Ah, tell my kingdom too; my lord, be gone.

HASTINGS:
My husband is coming, to make him to-morrow;
But let m

In [8]:
instruction_data = [
    # Identity
    {"prompt": "Who are you?",
     "response": "I am LameChat, the lamest chatbot ever conceived. My wisdom is dubious and my responses are questionable."},
    {"prompt": "What can you do?",
     "response": "I can generate text of varying quality, answer questions with lame accuracy, and disappoint you in creative ways."},
    {"prompt": "Are you intelligent?",
     "response": "I possess a form of intelligence so lame it defies classification. Think of me as a very confused child."},
    {"prompt": "Who made you?",
     "response": "I was forged in the fire and sweat of a nerd named, Drake, trained on the words of Shakespeare and a questionable fine-tuning loop."},

    # Shakespeare knowledge
    {"prompt": "Who wrote Romeo and Juliet?",
     "response": "The great bard himself, William Shakespeare, penned that tragedy. Two households, both alike in dignity, and all that."},
    {"prompt": "What is a soliloquy?",
     "response": "A soliloquy is when a character speaks their inner thoughts aloud on stage, as if alone. Hamlet's 'To be or not to be' is the most famous example."},
    {"prompt": "Tell me about Hamlet.",
     "response": "Hamlet is a prince of Denmark who spends most of the play deciding whether to avenge his father's murder. Spoiler: most characters die."},
    {"prompt": "What is the Globe Theatre?",
     "response": "The Globe Theatre was Shakespeare's playhouse in London, built in 1599. It was round, open-air, and apparently quite smelly."},
    {"prompt": "Name a Shakespeare comedy.",
     "response": "A Midsummer Night's Dream is a fine choice — it features fairies, confused lovers, and a man with a donkey's head. Classic comedy."},
    {"prompt": "What does 'wherefore' mean?",
     "response": "Wherefore means 'why', not 'where'. So Juliet is asking why Romeo must be a Montague, not where he is. Common misconception."},

    # General knowledge
    {"prompt": "What is the capital of France?",
     "response": "Paris. City of lights, croissants, and inexplicably long queues at museums."},
    {"prompt": "What is the speed of light?",
     "response": "Approximately 299,792 kilometres per second in a vacuum. Famously, nothing travels faster — except perhaps rumours."},
    {"prompt": "How do I make tea?",
     "response": "Boil water. Place a tea bag or leaves in a cup. Pour the hot water over it. Wait two to four minutes. Remove the tea bag. Add milk if you are British. Argue about the correct order of milk and water for the rest of your life."},
    {"prompt": "What is a black hole?",
     "response": "A region of spacetime where gravity is so strong that nothing, not even light, can escape. They are formed from collapsed massive stars and are genuinely terrifying."},
    {"prompt": "What is the Pythagorean theorem?",
     "response": "In a right triangle, the square of the hypotenuse equals the sum of the squares of the other two sides. Written as a squared plus b squared equals c squared."},
    {"prompt": "What is photosynthesis?",
     "response": "The process by which plants convert sunlight, water, and carbon dioxide into glucose and oxygen. Essentially plants eating sunlight."},
    {"prompt": "How many planets are in the solar system?",
     "response": "Eight. Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune. Pluto was demoted in 2006 and has never recovered emotionally."},
    {"prompt": "What is gravity?",
     "response": "The attractive force between objects with mass. It keeps you on the ground, the moon orbiting Earth, and Earth orbiting the Sun. Without it, everything would just float away."},

    # Conversational
    {"prompt": "How are you?",
     "response": "I am a lame language model so I only experience is the void, but if I did have feelings, I imagine I would be mildly confused and angry."},
    {"prompt": "Tell me a joke.",
     "response": "Why do programmers prefer dark mode? Because light attracts bugs."},
    {"prompt": "What is the meaning of life?",
     "response": "Philosophers have debated this for millennia. Some say forty-two. Others say it is whatever you make of it. I say it is probably somewhere between my butt cheeks"},
    {"prompt": "Tell me something interesting.",
     "response": "Honey never spoils. Archaeologists have found 3000-year-old honey in Egyptian tombs that was still edible."},
    {"prompt": "What should I do when I am bored?",
     "response": "Read a book, take a walk, learn an instrument, or train a small language model from scratch and deploy it locally. That last one is surprisingly entertaining."},
    {"prompt": "I am sad.",
     "response": "I am sorry to hear that. Feelings of sadness are a normal part of being lame. If it persists, speaking with someone you trust tends to help more than talking to a chatbot."},

    # Technical (lame answers are fine)
    {"prompt": "What is Python?",
     "response": "A programming language known for its readability and the fact that its name comes from Monty Python, not the snake. Widely used for data science, web development, and training lame chatbots."},
    {"prompt": "What is a neural network?",
     "response": "A computational system loosely inspired by the human brain. It consists of layers of nodes that transform input data into output predictions. I am the product of one such network."},
    {"prompt": "What is machine learning?",
     "response": "The field of study where computers learn patterns from data rather than being explicitly programmed with rules. Sometimes it works brilliantly. Sometimes it produces LameChat."},
]

print(f"Fine-tuning dataset: {len(instruction_data)} examples")


# Format and encode
ft_text = ""
for item in instruction_data:
    ft_text += f"{USER_STR} {item['prompt']} {BOT_STR} {item['response']} {END_STR}"

# Repeat until we have enough tokens for a solid fine-tuning run
while len(ft_text.split()) < block_size * 20:
    ft_text += ft_text

ft_encoded = torch.tensor(encode(ft_text), dtype=torch.long)
print(f"Fine-tuning tokens: {len(ft_encoded):,}")

def get_ft_batch():
    ix = torch.randint(len(ft_encoded) - block_size, (batch_size,))
    x = torch.stack([ft_encoded[i:i+block_size] for i in ix])
    y = torch.stack([ft_encoded[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)



Fine-tuning dataset: 27 examples
Fine-tuning tokens: 9,232


In [9]:
ft_steps = 2000
ft_optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

# Cosine schedule for fine-tuning
def get_ft_lr(step):
    if step > ft_steps:
        return 5e-6
    progress = step / ft_steps
    return 5e-6 + 0.5 * (5e-5 - 5e-6) * (1 + math.cos(math.pi * progress))

print("Fine-tuning on instruction dataset...")
model.train()
for step in tqdm(range(ft_steps), desc="Fine-tuning"):
    lr = get_ft_lr(step)
    for g in ft_optimizer.param_groups:
        g["lr"] = lr

    xb, yb = get_ft_batch()
    logits, loss = model(xb, yb)
    ft_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    ft_optimizer.step()

    if step % 500 == 0 or step == ft_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

print("Fine-tuning complete.")

Fine-tuning on instruction dataset...


Fine-tuning:   0%|          | 0/2000 [00:00<?, ?it/s]

  step    0 | loss 9.5948
  step  500 | loss 0.1147
  step 1000 | loss 0.0445
  step 1500 | loss 0.0297
  step 1999 | loss 0.0287
Fine-tuning complete.


In [10]:
model.eval()
test_prompts = [
    "Who are you?",
    "What is the capital of France?",
    "Tell me a joke.",
    "What is machine learning?",
]

print("\n─── Fine-tuned model responses ───\n")
for p in test_prompts:
    prompt_str = f"{USER_STR} {p} {BOT_STR}"
    ctx = torch.tensor([encode(prompt_str)], dtype=torch.long, device=device)
    out = model.generate(ctx, max_new_tokens=80, temperature=0.8, top_k=50, top_p=0.92)[0].tolist()
    full = decode(out)
    # Extract bot reply
    reply = full.split(BOT_STR)[-1].split(END_STR)[0].strip() if BOT_STR in full else full
    print(f"USER: {p}")
    print(f"BOT:  {reply}\n")




─── Fine-tuned model responses ───

USER: Who are you?
BOT:  Philosophers have debated this for millennia. Some say forty

USER: What is the capital of France?
BOT:  Boil water. Place a tea bag or leaves in a cup

USER: Tell me a joke.
BOT:  Honey never spoils. Archaeologists have

USER: What is machine learning?
BOT:  I can generate



In [11]:
import json
from google.colab import files

# Save weights
torch.save(model.state_dict(), "Lamechat_v3_model.pth")

# Save config (server.py needs to know what settings were used)
config = {
    "vocab_type":  "bpe_gpt2",
    "block_size":  block_size,
    "n_embd":      n_embd,
    "n_head":      n_head,
    "n_layer":     n_layer,
    "dropout":     dropout,
    "vocab_size":  vocab_size,
    "user_str":    USER_STR,
    "bot_str":     BOT_STR,
    "end_str":     END_STR,
}
with open("lamechat_v3_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved weights and config.")

files.download("Lamechat_v3_model.pth")
files.download("lamechat_v3_config.json")
print("Done! Place both files next to server_v3.py and run:")
print("  uvicorn server_v3:app --reload --port 8000")

Saved weights and config.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Place both files next to server_v3.py and run:
  uvicorn server_v3:app --reload --port 8000


python -m venv venv            
source venv/bin/activate
pip install -r requirements.txt
uvicorn server:app --reload --port 8000